# Market Data Loader — Yahoo Finance + Binance to local Parquet

Canonical loader for the trading system. Fetches OHLCV history, normalises it to one schema,
and persists it as Parquet under `store/` so later work (correlation study, backtests) reads
from disk instead of re-downloading.

**Supported now:** Forex (Yahoo) and Crypto (Yahoo or Binance). Equities are a one-line
addition later — the symbol mapper is the only asset-class-aware piece.

Scope: **history to disk** only. Live streaming stays in
`development/volume/dev-tradingsystem.ipynb` for now.

## Store layout

```
development/marketdata/store/{asset_class}/{symbol}/{timeframe}/{source}.parquet
```

One file per source, so two feeds for the same instrument never overwrite each other and can
be compared — which matters when checking Yahoo FX against a real broker feed later.

Canonical symbols are source-agnostic; the mapper translates per feed:

| Asset class | Canonical  | Yahoo      | Binance     |
|-------------|------------|------------|-------------|
| forex       | `EURUSD`   | `EURUSD=X` | unsupported |
| crypto      | `BTC-USD`  | `BTC-USD`  | `BTCUSD`    |
| crypto      | `BTC-USDT` | no such quote | `BTCUSDT` |

In [ ]:
# !pip install yfinance pandas requests pyarrow
from __future__ import annotations

import time
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Literal

import numpy as np
import pandas as pd
import requests
import yfinance as yf

STORE_ROOT = Path("store")
OHLCV = ["open", "high", "low", "close", "volume"]

AssetClass = Literal["forex", "crypto"]
Source = Literal["yahoo", "binance"]

## Instrument and symbol mapping

`Instrument` is the key for everything downstream: it names the file on disk and it is what
you hand the loader. `timeframe` is part of the identity, so one symbol can be stored at
several resolutions side by side.

In [ ]:
@dataclass(frozen=True)
class Instrument:
    """One thing to fetch and store.

    symbol: canonical and source-agnostic — "EURUSD" for forex, "BTC-USDT" for crypto.
    timeframe: Binance-style token (1m, 5m, 15m, 1h, 4h, 1d, 1w).
    """
    symbol: str
    asset_class: AssetClass
    source: Source = "yahoo"
    timeframe: str = "1d"

    def __str__(self) -> str:
        return f"{self.symbol}@{self.source}:{self.timeframe}"


def to_source_symbol(inst: Instrument) -> str:
    """Translate a canonical symbol into what the source expects."""
    if inst.source == "yahoo":
        if inst.asset_class == "forex":
            return f"{inst.symbol}=X"           # EURUSD -> EURUSD=X
        return inst.symbol                       # BTC-USD stays as-is
    if inst.source == "binance":
        if inst.asset_class != "crypto":
            raise ValueError(
                f"Binance serves crypto only; {inst.symbol} is {inst.asset_class}. "
                "Use source='yahoo' for forex."
            )
        return inst.symbol.replace("-", "")       # BTC-USDT -> BTCUSDT
    raise ValueError(f"unknown source {inst.source!r}")


def _ts(value) -> pd.Timestamp:
    """Accept '2024-01-01', '20240101' or a Timestamp; always return UTC."""
    t = pd.Timestamp(value)
    return t.tz_localize("UTC") if t.tzinfo is None else t.tz_convert("UTC")

## Feeds

Both feeds return the same object: a `DataFrame` indexed by UTC timestamp with columns
`open, high, low, close, volume`, float dtype, sorted, no duplicate index entries.

Two Yahoo quirks to know before trusting the output:

- **Intraday history is capped** — 1-minute bars reach back 7 days, most other sub-daily
  intervals 60 days, hourly 730 days. Asking for more returns a short frame rather than an
  error, so `YahooFeed` warns when the request exceeds the cap.
- **Forex volume is meaningless** — Yahoo reports 0 for FX bars because there is no central
  exchange. Volume-based signals need a broker feed or a futures proxy instead.

In [ ]:
class YahooFeed:
    """Yahoo Finance history. Forex and crypto (equities work unchanged)."""

    # timeframes Yahoo has no native bar for -> (fetch at, resample to)
    _RESAMPLE_FALLBACK = {"4h": ("1h", "4h"), "3d": ("1d", "3D"), "1w": ("1d", "1W")}
    # how far back each interval is available, in days
    _MAX_LOOKBACK_DAYS = {"1m": 7, "2m": 60, "5m": 60, "15m": 60, "30m": 60, "90m": 60, "1h": 730}

    def history(self, inst: Instrument, start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
        fetch_tf, resample_rule = self._RESAMPLE_FALLBACK.get(inst.timeframe, (inst.timeframe, None))
        self._warn_if_beyond_lookback(fetch_tf, start)

        raw = yf.Ticker(to_source_symbol(inst)).history(
            start=start, end=end, interval=fetch_tf, auto_adjust=False,
        )
        if raw.empty:
            return _empty_ohlcv()

        df = raw.rename(columns=str.lower)[OHLCV].astype(float)
        df.index = pd.to_datetime(df.index, utc=True)

        if resample_rule:
            df = df.resample(resample_rule).agg(
                {"open": "first", "high": "max", "low": "min", "close": "last", "volume": "sum"}
            ).dropna(how="all")

        return _normalise(df)

    def _warn_if_beyond_lookback(self, interval: str, start: pd.Timestamp) -> None:
        cap = self._MAX_LOOKBACK_DAYS.get(interval)
        if cap is None:
            return
        requested = (pd.Timestamp.now(tz="UTC") - start).days
        if requested > cap:
            print(f"  ! Yahoo serves at most {cap}d of {interval} bars; "
                  f"asked for {requested}d. Expect a truncated frame.")


class BinanceFeed:
    """Binance spot klines. Crypto only. Pages through the 1000-bar REST limit."""

    _REST = "https://api.binance.com/api/v3/klines"
    _COLS = ["open_time", "open", "high", "low", "close", "volume", "close_time",
             "quote_volume", "trades", "taker_base_vol", "taker_quote_vol", "ignore"]
    _PAGE_LIMIT = 1000

    def history(self, inst: Instrument, start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
        symbol = to_source_symbol(inst)
        start_ms, end_ms = int(start.timestamp() * 1000), int(end.timestamp() * 1000)
        frames = []

        while start_ms < end_ms:
            resp = requests.get(self._REST, params={
                "symbol": symbol, "interval": inst.timeframe,
                "startTime": start_ms, "endTime": end_ms, "limit": self._PAGE_LIMIT,
            }, timeout=30)
            resp.raise_for_status()
            batch = resp.json()
            if not batch:
                break
            frames.append(pd.DataFrame(batch, columns=self._COLS))
            start_ms = batch[-1][6] + 1           # resume after last close_time
            if len(batch) < self._PAGE_LIMIT:
                break
            time.sleep(0.25)                       # stay inside the request weight limit

        if not frames:
            return _empty_ohlcv()

        df = pd.concat(frames, ignore_index=True)
        df.index = pd.to_datetime(df["open_time"], unit="ms", utc=True)
        return _normalise(df[OHLCV].astype(float))


def _empty_ohlcv() -> pd.DataFrame:
    idx = pd.DatetimeIndex([], tz="UTC", name="timestamp")
    return pd.DataFrame(columns=OHLCV, index=idx, dtype=float)


def _normalise(df: pd.DataFrame) -> pd.DataFrame:
    """One schema for every feed: UTC index named 'timestamp', sorted, de-duplicated."""
    df = df[~df.index.duplicated(keep="last")].sort_index()
    df.index.name = "timestamp"
    return df[OHLCV].astype(float)

## Parquet store

Writes are **incremental**. An existing file is read, merged with the new rows, de-duplicated
on the index keeping the newest copy, then rewritten. Re-running a download over an
overlapping date range therefore extends the file instead of clobbering it, and a revised bar
replaces the stale one.

In [ ]:
class ParquetStore:
    """Local Parquet store, one file per (asset_class, symbol, timeframe, source)."""

    def __init__(self, root: Path | str = STORE_ROOT):
        self.root = Path(root)

    def path(self, inst: Instrument) -> Path:
        return self.root / inst.asset_class / inst.symbol / inst.timeframe / f"{inst.source}.parquet"

    def read(self, inst: Instrument) -> pd.DataFrame:
        p = self.path(inst)
        if not p.exists():
            return _empty_ohlcv()
        df = pd.read_parquet(p)
        df.index = pd.to_datetime(df.index, utc=True)
        return _normalise(df)

    def write(self, inst: Instrument, df: pd.DataFrame) -> Path:
        """Merge `df` into whatever is already stored; return the file path."""
        p = self.path(inst)
        if df.empty:
            return p

        merged = _normalise(pd.concat([self.read(inst), df]) if p.exists() else df)
        p.parent.mkdir(parents=True, exist_ok=True)
        merged.to_parquet(p, engine="pyarrow", compression="snappy")
        return p

    def coverage(self) -> pd.DataFrame:
        """What is on disk: one row per stored file."""
        rows = []
        for p in sorted(self.root.rglob("*.parquet")):
            asset_class, symbol, timeframe = p.parts[-4:-1]
            df = pd.read_parquet(p)
            rows.append({
                "asset_class": asset_class, "symbol": symbol, "timeframe": timeframe,
                "source": p.stem, "rows": len(df),
                "start": df.index.min(), "end": df.index.max(),
                "size_kb": round(p.stat().st_size / 1024, 1),
            })
        return pd.DataFrame(rows)

## Loader

`download()` fetches and persists, `load()` reads back from disk, and `panel()` builds the
wide one-column-per-instrument frame that the correlation and cointegration work needs.

A failing symbol is recorded in the summary rather than aborting the batch — one delisted or
mistyped ticker should not cost you the whole download.

In [ ]:
class MarketDataLoader:
    """Fetch history from any supported source, store it, read it back."""

    def __init__(self, store: ParquetStore | None = None):
        self.store = store or ParquetStore()
        self._feeds = {"yahoo": YahooFeed(), "binance": BinanceFeed()}

    def download(self, instruments: Iterable[Instrument], start, end,
                 verbose: bool = True) -> pd.DataFrame:
        """Fetch each instrument and merge it into the store. Returns a summary table."""
        start_ts, end_ts = _ts(start), _ts(end)
        summary = []

        for inst in instruments:
            if verbose:
                print(f"- {inst}")
            try:
                df = self._feeds[inst.source].history(inst, start_ts, end_ts)
                path = self.store.write(inst, df)
                summary.append({
                    "instrument": str(inst), "rows": len(df),
                    "start": df.index.min() if len(df) else pd.NaT,
                    "end": df.index.max() if len(df) else pd.NaT,
                    "status": "ok" if len(df) else "empty",
                    "path": str(path),
                })
            except Exception as exc:               # one bad symbol must not kill the batch
                summary.append({
                    "instrument": str(inst), "rows": 0, "start": pd.NaT, "end": pd.NaT,
                    "status": f"{type(exc).__name__}: {exc}"[:120], "path": "",
                })
                if verbose:
                    print(f"  ! {type(exc).__name__}: {exc}")

        return pd.DataFrame(summary)

    def load(self, inst: Instrument, start=None, end=None) -> pd.DataFrame:
        """Read one instrument back from the store, optionally sliced."""
        df = self.store.read(inst)
        if start is not None:
            df = df[df.index >= _ts(start)]
        if end is not None:
            df = df[df.index <= _ts(end)]
        return df

    def panel(self, instruments: Iterable[Instrument], field: str = "close",
              how: str = "inner") -> pd.DataFrame:
        """Wide frame, one column per instrument.

        `how="inner"` keeps only timestamps present for every instrument, which is what a
        correlation or cointegration study needs.
        """
        series = {}
        for inst in instruments:
            df = self.store.read(inst)
            if not df.empty:
                series[inst.symbol] = df[field]
        if not series:
            return pd.DataFrame()
        return pd.concat(series, axis=1, join=how).dropna(how="all")

## Forex

The seven USD majors plus three crosses. This is the universe the correlation study needs:
the crosses are algebraically implied by the majors, and measuring exactly that structure is
the point of the first study.

In [ ]:
loader = MarketDataLoader()

FX_MAJORS = ["EURUSD", "GBPUSD", "USDJPY", "USDCHF", "AUDUSD", "USDCAD", "NZDUSD"]
FX_CROSSES = ["EURGBP", "EURJPY", "AUDNZD"]

fx = [Instrument(s, "forex", "yahoo", "1d") for s in FX_MAJORS + FX_CROSSES]

fx_summary = loader.download(fx, start="2019-01-01", end="2026-09-10")
fx_summary

## Crypto

Binance for USDT pairs (deeper history, real volume), Yahoo for a USD-quoted cross-check.

In [ ]:
crypto = [
    Instrument("BTC-USDT", "crypto", "binance", "1h"),
    Instrument("ETH-USDT", "crypto", "binance", "1h"),
    Instrument("BTC-USD", "crypto", "yahoo", "1d"),
]

crypto_summary = loader.download(crypto, start="2025-01-01", end="2026-09-10")
crypto_summary

## What is on disk

In [ ]:
loader.store.coverage()

## Read back and build a panel

The panel is the input for the currency-factor decomposition. With
`log P(i,j) = s_i - s_j`, the covariance of these columns has rank at most
(number of currencies - 1); whatever is left over is the residual worth testing for mean
reversion.

In [ ]:
fx_panel = loader.panel(fx, field="close")
print(fx_panel.shape)
fx_panel.tail()

In [ ]:
# Sanity check: daily log returns and their correlation.
# Expect block structure driven by the shared USD leg, not ten independent assets.
fx_returns = np.log(fx_panel).diff().dropna()
fx_returns.corr().round(2)

## Limits to remember

- **Yahoo FX carries no bid/ask and reports zero volume.** Close prices are fine for a
  structure study; anything depending on spread has to wait for a broker feed.
- **Yahoo intraday history is capped** — 1m: 7 days, most sub-daily: 60 days, hourly: 730
  days. Deep intraday FX needs a dedicated source such as Dukascopy or HistData.
- **Binance is crypto-only**, and USDT is not USD — never mix `BTC-USDT` and `BTC-USD` in one
  series.
- Re-running `download()` over an overlapping range is safe: rows merge and de-duplicate.

Next: a broker feed for real FX spreads, then the cost model — spread, commission, and
overnight swap quoted separately for long and short — that every backtest depends on.